# Day 04 · Clusters — Where It Keeps Happening

**Question.** Which Chicago streets keep hurting the people who walk and bike on them?

Every bus or train trip starts and ends on foot. Plates I–III asked whether a rider can reach the ‘L’;
this plate looks at the walk itself.

**Approach.** Take every person walking or biking whom police recorded as killed or seriously injured
(KSI: the SR 1050 classes *fatal* and *incapacitating injury*) in the five years 2021–2025. Place each at
the crash location, find where they bunch together with HDBSCAN, and draw each cluster along the streets that join its people. Then
test the clusters against CDOT's High Injury Network (2025), the 100 miles of major streets the City has
named for safety work, and measure how many people walking were hurt next to a CTA bus stop.

**Output.** `output/plate.json`, read by the website to draw the plate.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, run settings |
| 2. Pull | Dated snapshots: people, crashes, Vision Zero fatalities, community areas; OSM streets, GTFS stops and city limits from `reference/` |
| 3. Validate | Schema, values and consistency checks that stop the run on surprises |
| 4. Transform | People joined to their crash, kept in the window and the city, projected |
| 5. High Injury Network | CDOT's 34 corridors drawn on the OSM street network |
| 6. Clusters | HDBSCAN, stability, street shapes, names |
| 7. Analyze | Summary, HIN and bus-stop tests, cross-checks, the finding |
| 8. QA view | Quick maps and charts to catch errors before publishing |
| 9. Export | `plate.json` with the data ledger |

Run top to bottom. To refresh the portal data, set `REFRESH = True` in section 1.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import pickle
import re
import subprocess
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import requests
import shapely
from scipy.spatial import cKDTree
from shapely.ops import nearest_points
from sklearn.cluster import HDBSCAN

from atlas import Ledger, snapshot, to_address, walk, write_plate

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day04')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull new snapshots from the Chicago Data Portal

DAY = 4
PEOPLE_ID = 'u6pd-qa9d'    # Traffic Crashes - People
CRASHES_ID = '85ca-t3if'   # Traffic Crashes - Crashes
VZ_ID = 'gzaz-isa6'        # Traffic Crashes - Vision Zero Chicago Traffic Fatalities
AREAS_ID = 'igwz-8jzy'     # Boundaries - Community Areas
PORTAL = 'https://data.cityofchicago.org'
CDOT_REPORT = ('https://api.chicago.gov/filenet5/servlets/getDocumentContent?applicationId=CompleteStreets'
               '&documentId=%7B805EE69A-0000-C518-8777-D32746FEA322%7D')  # Chicago Traffic Crashes, Fall 2025

# Paths: the notebook runs from its own folder (days/day04-clusters/)
DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
PROC_DIR = DAY_DIR / 'data' / 'processed'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
HIN_CSV = RAW_DIR / 'cdot_hin_2025.csv'
OSM_DIR = REPO / walk.OSM_DIR
GTFS_ZIP = sorted((REPO / 'reference' / 'raw' / 'gtfs').glob('cta_gtfs_*.zip'))[-1]
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
assert (DAY_DIR / 'day04.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

# Window: five full years. Citywide E-Crash records start in September 2017; the pull reaches back to
# 2018 for the sensitivity run.
YEARS = (2021, 2025)
PULL_FROM = '2018-01-01'
PULL_TO = '2025-12-31T23:59:59'
KSI = ('FATAL', 'INCAPACITATING INJURY')
MODES = {'PEDESTRIAN': 'walk', 'BICYCLE': 'bike'}
CRS = walk.CRS  # UTM 16N, meters

# Clusters -------------------------------------------------------------------------------------
MIN_CLUSTER = 12   # people in five years, about one every five months
MIN_SAMPLES = 6
SELECTION = 'leaf'  # 'eom' merges most of the South Side into one cluster
BOOT_RUNS = 200     # stability: refit on 80% subsamples
BOOT_FRACTION = 0.8
STABLE = 0.5        # Hennig: below 0.5 a cluster dissolves under resampling; those are not drawn
SHAPE_REACH_M = 1000  # street shapes: people joined only within this (weighted) distance; farther = a new piece
SIDE_STREET = 2.5     # street shapes prefer major streets: a side street counts this many times its length

# Tests ----------------------------------------------------------------------------------------
HIN_BUFFER_M = 22.86   # 75 ft, CDOT's buffer for assigning crashes to HIN segments
ON_HIN = 0.25          # a cluster is on the HIN when at least a quarter of its people were hurt on it
STOP_M = 30            # about 100 ft from a bus stop
MAJOR = {'trunk', 'primary', 'secondary', 'tertiary'}
NIGHT = {'DARKNESS', 'DARKNESS, LIGHTED ROAD'}
CDOT_2024 = {'walk': {'killed': 45, 'serious': 444}, 'bike': {'killed': 4, 'serious': 172}}  # report, p. 6-9
CDOT_HIN_MILES = 100   # report, p. 12

## 2. Pull

People and crashes come from the City's portal as dated snapshots. Only crashes whose most severe
injury is fatal or incapacitating are pulled: every person walking or biking who was killed or seriously
hurt is in one of them. The Vision Zero list is CDOT's reviewed count of traffic deaths (30-day rule,
expressways excluded); it decides who was killed (section 4).

In [ ]:
people_raw, pulled = snapshot(
    PEOPLE_ID, RAW_DIR, 'people_ksi', refresh=REFRESH,
    select='person_id,person_type,crash_record_id,crash_date,age,injury_classification',
    where=(f"person_type in('PEDESTRIAN','BICYCLE') and injury_classification in{KSI} "
           f"and crash_date between '{PULL_FROM}' and '{PULL_TO}'"),
)
CRASH_FIELDS = ('crash_record_id,crash_date,latitude,longitude,street_no,street_direction,street_name,'
                'lighting_condition,first_crash_type,posted_speed_limit,hit_and_run_i')
crashes_raw, crashes_pulled = snapshot(
    CRASHES_ID, RAW_DIR, 'crashes_ksi', refresh=REFRESH, select=CRASH_FIELDS,
    where=f"most_severe_injury in{KSI} and crash_date between '{PULL_FROM}' and '{PULL_TO}'",
)
# A few people are KSI in the people table while their crash's most-severe field says otherwise (one
# record updated, not the other): pull those crashes by id.
missing = sorted(set(people_raw['crash_record_id']) - set(crashes_raw['crash_record_id']))
if missing:
    extra, _ = snapshot(CRASHES_ID, RAW_DIR, 'extra_crashes', refresh=REFRESH, select=CRASH_FIELDS,
                        where=f"crash_record_id in({','.join(repr(m) for m in missing)})")
    log.info('%d crashes pulled by id', len(extra))
    crashes_raw = pd.concat([crashes_raw, extra], ignore_index=True).drop_duplicates('crash_record_id')
vz_raw, vz_pulled = snapshot(VZ_ID, RAW_DIR, 'vz_fatalities', refresh=REFRESH,
                             select='person_id,crash_date,crash_location,victim,latitude,longitude')
log.info('People %d, crashes %d, Vision Zero %d (pulled %s)', len(people_raw), len(crashes_raw), len(vz_raw), pulled)

areas_path = sorted(RAW_DIR.glob('community_areas_*.geojson'))
if REFRESH or not areas_path:
    resp = requests.get(f'{PORTAL}/resource/{AREAS_ID}.geojson', timeout=60)
    resp.raise_for_status()
    areas_path = [RAW_DIR / f'community_areas_{pulled.isoformat()}.geojson']
    areas_path[0].write_text(resp.text)
areas = gpd.read_file(areas_path[-1])[['community', 'geometry']].to_crs(CRS)
city = shapely.union_all(gpd.read_file(BOUNDARY_RAW).to_crs(CRS).geometry)
boundary_pulled = BOUNDARY_RAW.stem.rsplit('_', 1)[-1]

In [ ]:
# Streets: the drivable named network from the newest Geofabrik extract, cut to the city's box.
pbf, osm_date = walk.latest_pbf(OSM_DIR)
PROC_DIR.mkdir(parents=True, exist_ok=True)
streets_pbf = PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.osm.pbf'
if not streets_pbf.exists():
    box = PROC_DIR / f'chicago-{osm_date.isoformat()}.osm.pbf'
    subprocess.run(['osmium', 'extract', '-b', '-87.95,41.64,-87.52,42.03', str(pbf), '-o', str(box), '--overwrite'], check=True)
    subprocess.run(['osmium', 'tags-filter', str(box),
                    'w/highway=trunk,primary,secondary,tertiary,unclassified,residential,living_street',
                    '-o', str(streets_pbf), '--overwrite'], check=True)
streets = gpd.read_file(streets_pbf, layer='lines', columns=['name', 'highway']).to_crs(CRS)
log.info('OSM extract of %s: %d street ways', osm_date, len(streets))

with zipfile.ZipFile(GTFS_ZIP) as z:
    gtfs_stops = pd.read_csv(z.open('stops.txt'))
feed_date = GTFS_ZIP.stem.rsplit('_', 1)[-1]
bus_stops = gtfs_stops[gtfs_stops['stop_id'] < 30000]  # CTA: bus stops below 30000, rail above
hin_table = pd.read_csv(HIN_CSV)

## 3. Validate

In [ ]:
need_people = {'person_id', 'person_type', 'crash_record_id', 'crash_date', 'age', 'injury_classification'}
need_crashes = {'crash_record_id', 'latitude', 'longitude', 'street_name', 'lighting_condition', 'hit_and_run_i'}
checks = {
    'people columns': need_people <= set(people_raw.columns),
    'crash columns': need_crashes <= set(crashes_raw.columns),
    'people: walking or biking only': people_raw['person_type'].isin(MODES).all(),
    'people: KSI only': people_raw['injury_classification'].isin(KSI).all(),
    'people: one row per person': people_raw['person_id'].is_unique,
    'crashes: one row per crash': crashes_raw['crash_record_id'].is_unique,
    'every person has a crash': people_raw['crash_record_id'].isin(crashes_raw['crash_record_id']).all(),
    'HIN: 34 corridors': hin_table['corridor'].nunique() == 34,
    'bus stops present': len(bus_stops) > 10_000,
}
for name, ok in checks.items():
    log.info('%-32s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

## 4. Transform

One row per person, with the crash's location and conditions.

**Deaths come from CDOT's Vision Zero list**, not the portal's injury field. The field is set at the
scene and often not updated: of the people walking or biking on the list, many are coded *incapacitating*
in the portal and some are missing from its KSI set. Each death on the list is matched to a portal
person by id, else by mode, time (within two hours) and place (within a tenth of a mile); deaths with no
match are added at the list's location. People the police coded *fatal* who are not on the list (CDOT
excludes, for example, deaths from medical causes or intentional acts) stay in the count as seriously
injured.

People without coordinates, or placed outside the city limits, are counted and dropped: the plate draws
every person it counts.

In [ ]:
df = people_raw.merge(crashes_raw.drop(columns='crash_date'), on='crash_record_id', how='left')
df['date'] = pd.to_datetime(df['crash_date'])
df['year'] = df['date'].dt.year
df['mode'] = df['person_type'].map(MODES)
df['killed'] = df['injury_classification'].eq('FATAL')
df['age'] = pd.to_numeric(df['age'], errors='coerce').where(lambda a: a.between(1, 110))
for c in ('latitude', 'longitude'):
    df[c] = pd.to_numeric(df[c], errors='coerce')

window = df[df['year'].between(*YEARS)].copy()
window['killed_police'] = window['killed']

VZ_MODES = {'PEDESTRIAN': 'walk', 'CYCLIST': 'bike'}
vz = vz_raw.assign(date=pd.to_datetime(vz_raw['crash_date']), mode=vz_raw['victim'].map(VZ_MODES))
vz = vz[vz['date'].dt.year.between(*YEARS) & vz['mode'].notna()].copy()
for d in (vz, window):
    lat, lon = (pd.to_numeric(d[c], errors='coerce') for c in ('latitude', 'longitude'))
    d['latitude'], d['longitude'] = lat, lon
    d['ax'], d['ay'] = to_address(lat, lon)
# By id when the id and the day agree (the list reuses one id for two deaths and leaves some blank)
day_of = window.set_index('person_id')['date'].dt.normalize()
by_id = vz['person_id'].map(day_of).eq(vz['date'].dt.normalize())
match, how = {}, {}
for r in vz[by_id].itertuples():
    match[r.Index], how[r.Index] = r.person_id, 'id'
for r in vz[~by_id].itertuples():  # each portal person matches one death at most
    c = window[(window['mode'] == r.mode) & ((window['date'] - r.date).abs() <= pd.Timedelta(hours=2))
               & ~window['person_id'].isin(set(match.values()))]
    c = c[np.hypot(c['ax'] - r.ax, c['ay'] - r.ay) <= 80]  # a tenth of a mile
    if len(c):
        match[r.Index] = c.loc[(c['date'] - r.date).abs().idxmin(), 'person_id']
        how[r.Index] = 'time and place'
vz['matched'] = pd.Series(how)
window['killed'] = window['person_id'].isin(set(match.values()))


def street_of(location: str) -> str:
    """'4035 W 26TH ST' -> '26TH ST', the portal's street_name form."""
    words = re.sub(r'^\d+\s+[NSEW]\s+', '', str(location).upper()).split()
    return ' '.join({'STREET': 'ST', 'AVENUE': 'AVE', 'ROAD': 'RD', 'BOULEVARD': 'BLVD', 'DRIVE': 'DR'}.get(w, w) for w in words)


added = vz[vz['matched'].isna()].assign(
    person_id=lambda d: 'VZ-' + d.index.astype(str), killed=True, killed_police=False,
    year=lambda d: d['date'].dt.year, street_name=lambda d: d['crash_location'].map(street_of))
window = pd.concat([window, added[['person_id', 'date', 'year', 'mode', 'killed', 'killed_police', 'latitude',
                                    'longitude', 'street_name']]], ignore_index=True)
window['night'] = window['lighting_condition'].map(lambda v: v in NIGHT if isinstance(v, str) else np.nan).astype(float)
window['hit_run'] = window['hit_and_run_i'].eq('Y').astype(float)  # blank = not flagged
window.loc[window['person_id'].str.startswith('VZ-'), 'hit_run'] = np.nan  # unknown for added deaths
deaths = {
    'visionZero': len(vz), 'byId': int((vz['matched'] == 'id').sum()),
    'byTimePlace': int((vz['matched'] == 'time and place').sum()), 'added': len(added),
    'codedSerious': int((window['killed'] & ~window['killed_police'] & ~window['person_id'].str.startswith('VZ-')).sum()),
    'policeNotOnList': int((window['killed_police'] & ~window['killed']).sum()),
}
log.info('Deaths: %s', deaths)
assert deaths['byId'] + deaths['byTimePlace'] + deaths['added'] == deaths['visionZero']
assert int(window['killed'].sum()) == deaths['visionZero']

no_xy = window['latitude'].isna() | window['latitude'].eq(0)
pts = gpd.GeoSeries(gpd.points_from_xy(window['longitude'], window['latitude']), crs='EPSG:4326').to_crs(CRS)
inside = pd.Series(pts.within(city).to_numpy(), index=window.index) & ~no_xy
dropped = {'noCoordinates': int(no_xy.sum()), 'outsideCity': int((~inside & ~no_xy).sum())}
log.info('Window %s: %d people; dropped %s', YEARS, len(window), dropped)

ppl = window[inside].copy()
XY = np.column_stack([pts[inside.to_numpy()].x, pts[inside.to_numpy()].y])
ppl['x'], ppl['y'] = to_address(ppl['latitude'], ppl['longitude'])
ppl = ppl.reset_index(drop=True)
ppl.groupby(['mode', 'killed']).size().unstack()

## 5. High Injury Network

CDOT names its 34 corridors by street and cross streets (report p. 13) but does not publish them as
data. Each corridor is drawn on the OSM street network: the street's ways (all name variants), the
points where its two cross streets meet it, and the shortest path between them along the street.
Pieces of one street are joined across gaps under 500 m (a bridge under construction, a viaduct
mapped under another tag); a cross street that stops short of the street (a park, a T) meets it at its
nearest point. The total must come out near CDOT's "roughly 100 miles".

In [ ]:
SUFFIX = {'AVENUE': 'AVE', 'STREET': 'ST', 'ROAD': 'RD', 'BOULEVARD': 'BLVD', 'DRIVE': 'DR', 'COURT': 'CT',
          'PLACE': 'PL', 'PARKWAY': 'PKWY', 'DOCTOR': 'DR', 'JUNIOR': 'JR'}
ALIAS = {'DR MARTIN LUTHER KING JR DR': 'MARTIN LUTHER KING JR DR',
         'JEAN BAPTISTE POINT DUSABLE LAKE SHORE DR': 'DUSABLE LAKE SHORE DR', 'BROADWAY AVE': 'BROADWAY'}
GAP_M, LIMIT_SNAP_M = 500, 500


def norm(name) -> str:
    """'West North Avenue' -> 'NORTH AVE', 'North Broadway' -> 'BROADWAY' (direction dropped)."""
    if not isinstance(name, str):
        return ''
    words = re.sub(r'[^A-Z0-9 ]', '', name.upper()).split()
    if len(words) > 2 or (len(words) == 2 and words[1] not in {*SUFFIX, *SUFFIX.values()}):
        if words[0] in {'NORTH', 'SOUTH', 'EAST', 'WEST', 'N', 'S', 'E', 'W'}:
            words = words[1:]
    s = ' '.join(SUFFIX.get(w, w) for w in words)
    return ALIAS.get(s, s)


streets['key'] = streets['name'].map(norm)


def lines_of(names: str) -> gpd.GeoSeries:
    g = streets.loc[streets['key'].isin({norm(n) for n in names.split('|')}), 'geometry']
    assert len(g), names
    return g


def street_graph(geoms) -> nx.Graph:
    G = nx.Graph()
    for geom in geoms:
        for part in getattr(geom, 'geoms', [geom]):
            c = [tuple(np.round(p, 1)) for p in part.coords]
            for a, b in zip(c, c[1:]):
                G.add_edge(a, b, w=float(np.hypot(a[0] - b[0], a[1] - b[1])))
    for comp in [list(c) for c in nx.connected_components(G)]:  # join pieces of the same street
        others = [n for n in G if n not in set(comp)]
        if not others:
            break
        d, j = cKDTree(others).query(comp)
        i = int(np.argmin(d))
        if d[i] < GAP_M:
            G.add_edge(comp[i], others[j[i]], w=float(d[i]))
    return G


def meets(main: gpd.GeoSeries, cross: gpd.GeoSeries) -> list[shapely.Point]:
    m, x = shapely.union_all(main.values), shapely.union_all(cross.values)
    hit = m.intersection(x)
    found = [p for p in getattr(hit, 'geoms', [hit]) if p.geom_type == 'Point']
    if found:
        return found
    a, b = nearest_points(m, x)
    assert a.distance(b) < LIMIT_SNAP_M, (a.distance(b),)
    return [a]


hin_rows = []
for r in hin_table.itertuples():
    main = lines_of(r.street)
    G = street_graph(main)
    nodes = list(G.nodes)
    tree = cKDTree(nodes)
    best = None
    for pa in meets(main, lines_of(getattr(r, '_4'))):
        for pb in meets(main, lines_of(r.to)):
            na, nb = nodes[tree.query([pa.x, pa.y])[1]], nodes[tree.query([pb.x, pb.y])[1]]
            try:
                length, path = nx.single_source_dijkstra(G, na, nb, weight='w')
            except nx.NetworkXNoPath:
                continue
            if best is None or length < best[0]:
                best = (length, path)
    assert best, r
    hin_rows.append({'corridor': r.corridor, 'label': r.label, 'street': r.street,
                     'miles': best[0] / 1609.344, 'geometry': shapely.LineString(best[1])})
hin = gpd.GeoDataFrame(hin_rows, crs=CRS)
hin_miles = hin['miles'].sum()
log.info('HIN drawn: %d pieces, %.1f miles (CDOT: about %d)', len(hin), hin_miles, CDOT_HIN_MILES)
assert abs(hin_miles - CDOT_HIN_MILES) < 5, 'HIN length is off; check the corridor table'
hin.assign(miles=hin['miles'].round(2)).drop(columns='geometry')

## 6. Clusters

HDBSCAN on the people's positions in meters. `leaf` selection keeps street-scale clusters; the default
(`eom`) merges whole districts. Stability follows Hennig's clusterboot: refit on 80% subsamples and,
for each cluster, average the best Jaccard match; above 0.75 is stable, below 0.5 dissolves. Only
clusters at 0.5 or more are kept, ranked again by size; people in the others count as scattered. A grid
of settings (minimum 12–25, min samples 6–25, leaf and eom) gave no more stable result. Each kept
cluster is drawn on the streets: every person is placed at the nearest street corner of the OSM
network, and the cluster's shape is the minimum spanning tree of the network distances between them,
each tree edge drawn along its shortest path. Paths prefer major streets (a side street counts 2.5
times its length), and people more than 1 km apart are not joined, so a cluster may come in pieces. The shape follows the streets the people were hurt on
instead of cutting across blocks.

In [ ]:
def fit(X: np.ndarray, min_cluster: int = MIN_CLUSTER, min_samples: int = MIN_SAMPLES) -> np.ndarray:
    return HDBSCAN(min_cluster_size=min_cluster, min_samples=min_samples,
                   cluster_selection_method=SELECTION, copy=True).fit_predict(X)


raw_labels = fit(XY)
# rank clusters by people, largest first
sizes = pd.Series(raw_labels[raw_labels >= 0]).value_counts()
rank = {lab: i + 1 for i, lab in enumerate(sizes.index)}
ppl['cluster'] = [rank.get(l, 0) for l in raw_labels]  # 0 = not in a cluster
n_clusters = len(sizes)
log.info('%d clusters holding %.1f%% of people', n_clusters, 100 * (ppl['cluster'] > 0).mean())

rng = np.random.default_rng(4)
members = {c: set(np.flatnonzero(ppl['cluster'].to_numpy() == c)) for c in range(1, n_clusters + 1)}
jac = {c: [] for c in members}
for _ in range(BOOT_RUNS):
    idx = np.sort(rng.choice(len(XY), int(BOOT_FRACTION * len(XY)), replace=False))
    lab = fit(XY[idx])
    boot = [set(idx[lab == b]) for b in range(lab.max() + 1)]
    in_sample = set(idx)
    for c, m in members.items():
        m = m & in_sample
        jac[c].append(max((len(m & b) / len(m | b) for b in boot), default=0.0) if m else 0.0)
stability = {c: float(np.mean(v)) for c, v in jac.items()}
found = n_clusters
log.info('Stability: median %.2f; stable (>0.75) %d of %d', np.median(list(stability.values())),
         sum(v > 0.75 for v in stability.values()), found)

# keep the clusters that hold up, rank them again by size
kept = [c for c in range(1, found + 1) if stability[c] >= STABLE]
rerank = {old: new for new, old in enumerate(kept, start=1)}
ppl['cluster'] = ppl['cluster'].map(lambda c: rerank.get(c, 0))
stability = {rerank[c]: stability[c] for c in kept}
n_clusters = len(kept)
log.info('Kept %d of %d clusters, holding %.1f%% of people', n_clusters, found, 100 * (ppl['cluster'] > 0).mean())

In [ ]:
def pretty(street: str) -> str:
    """'CICERO AVE' -> 'Cicero', '79TH ST' -> '79th', 'DR MARTIN LUTHER KING JR DR' -> 'King Dr'."""
    if 'MARTIN LUTHER KING' in street:
        return 'King Dr'
    words = street.split()
    if len(words) > 1 and words[-1] in SUFFIX.values():
        words = words[:-1]
    return ' '.join(w.lower() if re.fullmatch(r'\d+(ST|ND|RD|TH)', w) else w.title() for w in words)


area_of = gpd.sjoin(gpd.GeoDataFrame(geometry=gpd.points_from_xy(*XY.T), crs=CRS), areas,
                    predicate='within', how='left')['community']
ppl['community'] = area_of.groupby(level=0).first().reindex(ppl.index).str.title().str.replace("O'hare", "O'Hare")
on_hin_pt = shapely.union_all(hin.geometry.buffer(HIN_BUFFER_M).values)
ppl['onHin'] = shapely.contains_xy(on_hin_pt, XY[:, 0], XY[:, 1])

# Street graph for the shapes: the same OSM streets as an undirected network, cached in processed/.
graph_pkl = PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.graph.pkl'
if graph_pkl.exists():
    G = pickle.loads(graph_pkl.read_bytes())
else:
    xml = PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.osm'
    if not xml.exists():
        subprocess.run(['osmium', 'cat', str(streets_pbf), '-o', str(xml), '--overwrite'], check=True)
    G = ox.convert.to_undirected(ox.project_graph(ox.graph_from_xml(xml, simplify=True, retain_all=False), to_crs=CRS))
    graph_pkl.write_bytes(pickle.dumps(G))
# path weight: length, with side streets counted longer so shapes follow the arterials people were hurt on
for u, v, k, d in G.edges(keys=True, data=True):
    hw = d.get('highway')
    d['w'] = d['length'] * (1 if set(hw if isinstance(hw, list) else [hw]) & MAJOR else SIDE_STREET)
g_nodes = list(G.nodes)
snap_m, snap_i = cKDTree(np.array([[G.nodes[n]['x'], G.nodes[n]['y']] for n in g_nodes])).query(XY)
ppl['node'] = [g_nodes[i] for i in snap_i]
log.info('Street graph: %d corners; people to nearest corner: median %.0f m, 95th %.0f m',
         len(g_nodes), np.median(snap_m), np.percentile(snap_m, 95))


def edge_line(u, v) -> shapely.LineString:
    d = min(G.get_edge_data(u, v).values(), key=lambda e: e['w'])
    return d.get('geometry') or shapely.LineString([(G.nodes[u]['x'], G.nodes[u]['y']), (G.nodes[v]['x'], G.nodes[v]['y'])])


def street_shape(corners) -> shapely.Geometry:
    """The streets joining a cluster's people: the minimum spanning tree of the network distances between
    their corners (side streets weighted longer), each tree edge drawn along its shortest path."""
    corners = sorted(set(corners))
    tree, paths = nx.Graph(), {}
    tree.add_nodes_from(corners)
    for n in corners:
        d, p = nx.single_source_dijkstra(G, n, cutoff=SHAPE_REACH_M, weight='w')
        for m in corners:
            if m != n and m in d:
                tree.add_edge(n, m, weight=d[m])
                paths[(n, m)] = p[m]
    lines = [edge_line(u, v) for a, b in nx.minimum_spanning_edges(tree, data=False)
             for pth in [paths.get((a, b)) or paths[(b, a)]] for u, v in zip(pth, pth[1:])]
    return shapely.line_merge(shapely.union_all(lines)) if lines else shapely.LineString()


rows, shapes = [], {}
for c in range(1, n_clusters + 1):
    g = ppl[ppl['cluster'] == c]
    xy = XY[g.index]
    counts = g['street_name'].value_counts()
    top = counts.index[:2].tolist()
    name = pretty(top[0]) if len(top) < 2 or counts.iloc[1] < 0.25 * len(g) else f'{pretty(top[0])} & {pretty(top[1])}'
    shapes[c] = street_shape(g['node'])
    rows.append({
        'rank': c, 'name': name, 'community': g['community'].mode().iloc[0], 'people': len(g),
        'walk': int(g['mode'].eq('walk').sum()), 'bike': int(g['mode'].eq('bike').sum()),
        'killed': int(g['killed'].sum()), 'night': g['night'].mean(), 'hitRun': g['hit_run'].mean(),
        'hinShare': g['onHin'].mean(), 'onHin': bool(g['onHin'].mean() >= ON_HIN),
        'stability': stability[c], 'miles': shapes[c].length / 1609.344,
    })
clusters = pd.DataFrame(rows)
# every person in a cluster should stand near its shape (their corner is on it unless out of reach)
near = [shapely.distance(shapes[c], shapely.points(XY[ppl['cluster'] == c])) for c in shapes]
log.info('People within 150 m of their cluster shape: %.1f%%', 100 * np.mean(np.concatenate(near) <= 150))
assert np.mean(np.concatenate(near) <= 150) > 0.95, 'Cluster shapes miss their people'
clusters.head(15).round(2)

In [ ]:
# Sensitivity: smaller and larger minimum clusters, and the eight years 2018-2025 (minimum scaled to
# the same rate, 12 people per five years).
def cluster_stats(X: np.ndarray, min_cluster: int, min_samples: int) -> dict:
    lab = fit(X, min_cluster, min_samples)
    return {'clusters': int(lab.max() + 1), 'share': float((lab >= 0).mean())}


sens = {f'minimum {m}': cluster_stats(XY, m, s) for m, s in [(10, 5), (15, 8)]}
long = df[df['year'].between(2018, 2025)].dropna(subset=['latitude'])
long = long[long['latitude'] > 0]
lp = gpd.GeoSeries(gpd.points_from_xy(long['longitude'], long['latitude']), crs='EPSG:4326').to_crs(CRS)
lp = lp[lp.within(city).to_numpy()]
sens['2018-2025'] = {**cluster_stats(np.column_stack([lp.x, lp.y]), 19, 10), 'people': int(len(lp))}
sens

## 7. Analyze

In [ ]:
P = len(ppl)
walking = ppl[ppl['mode'] == 'walk']

# Bus stops: people walking hurt within 30 m of a CTA bus stop, against the share of the street
# network that close to a stop (without it the share means nothing: stops sit at most arterial corners).
stop_xy = gpd.GeoSeries(gpd.points_from_xy(bus_stops['stop_lon'], bus_stops['stop_lat']),
                        crs='EPSG:4326').to_crs(CRS).get_coordinates().to_numpy()
d_stop, _ = cKDTree(stop_xy).query(XY)
ppl['nearStop'] = d_stop <= STOP_M
city_streets = streets[streets.intersects(city)].clip(city)
stop_zone = shapely.union_all(shapely.buffer(shapely.points(stop_xy), STOP_M))
street_len = city_streets.length.sum()
near_len = city_streets.intersection(stop_zone).length
street_near_stop = near_len.sum() / street_len
is_major = city_streets['highway'].isin(MAJOR)
major_near_stop = near_len[is_major].sum() / city_streets.length[is_major].sum()

# Major streets (OSM trunk to tertiary), the base of CDOT's HIN share
major = shapely.union_all(city_streets[city_streets['highway'].isin(MAJOR)].geometry.buffer(HIN_BUFFER_M).values)
ppl['onMajor'] = shapely.contains_xy(major, XY[:, 0], XY[:, 1])
in_c = ppl['cluster'] > 0
top10 = clusters.head(10)

summary = {
    'people': P, 'killed': int(ppl['killed'].sum()),
    'walk': int((ppl['mode'] == 'walk').sum()), 'bike': int((ppl['mode'] == 'bike').sum()),
    'walkKilled': int(walking['killed'].sum()), 'bikeKilled': int(ppl.loc[ppl['mode'] == 'bike', 'killed'].sum()),
    'clustersFound': found, 'clusters': n_clusters, 'inClusters': int(in_c.sum()), 'inClustersShare': in_c.mean(),
    'clusterMiles': float(shapely.union_all(list(shapes.values())).length / 1609.344),
    'night': ppl['night'].mean(), 'hitRun': ppl['hit_run'].mean(),
    'walk65': (walking['age'] >= 65).sum() / walking['age'].notna().sum(),
    'walkNearStop': ppl.loc[ppl['mode'] == 'walk', 'nearStop'].mean(), 'streetNearStop': street_near_stop,
    'majorNearStop': major_near_stop,
    'walkOnMajorNearStop': ppl.loc[(ppl['mode'] == 'walk') & ppl['onMajor'], 'nearStop'].mean(),
    'onMajor': ppl['onMajor'].mean(), 'onHin': ppl['onHin'].mean(),
    'onHinOfMajor': ppl.loc[ppl['onMajor'], 'onHin'].mean(),
    'hinMiles': hin_miles,
    'top10OnHin': int(top10['onHin'].sum()),
    'top10OnHinPeople': float(ppl.loc[ppl['cluster'].between(1, 10), 'onHin'].mean()),
    'clustersOnHin': int(clusters['onHin'].sum()),
    'stableClusters': int((clusters['stability'] > 0.75).sum()),
    **{f'deaths{k[0].upper()}{k[1:]}': v for k, v in deaths.items()},
}
log.info('Summary: %s', {k: round(v, 3) if isinstance(v, float) else v for k, v in summary.items()})
top10[['rank', 'name', 'community', 'people', 'walk', 'bike', 'killed', 'hinShare', 'onHin', 'stability']].round(2)

In [ ]:
# Cross-checks. Deaths by year: the police field against CDOT's Vision Zero list (now the count used);
# 2024 against CDOT's annual report, which uses IDOT's dataset (scooters and doorings added).
cross = pd.DataFrame({
    'police walk': window[window['killed_police'] & (window['mode'] == 'walk')].groupby('year').size(),
    'VZ walk': vz[vz['mode'] == 'walk'].groupby(vz['date'].dt.year).size(),
    'police bike': window[window['killed_police'] & (window['mode'] == 'bike')].groupby('year').size(),
    'VZ bike': vz[vz['mode'] == 'bike'].groupby(vz['date'].dt.year).size(),
}).fillna(0).astype(int)
y24 = window[window['year'] == 2024]
report = pd.DataFrame({
    'here': [int(((y24['mode'] == m) & (y24['killed'] == k)).sum()) for m in ('walk', 'bike') for k in (True, False)],
    'CDOT report': [CDOT_2024[m][k] for m in ('walk', 'bike') for k in ('killed', 'serious')],
}, index=[f'{m} {k}' for m in ('walk', 'bike') for k in ('killed', 'serious')])
display(cross, report)

In [ ]:
# The finding printed on the plate, generated from the data so it can never drift from the numbers.
def share(v: float) -> str:
    """A share in words when close to a simple fraction, else a percentage (as the registry)."""
    words = [(1 / 4, 'a quarter'), (1 / 3, 'one in three'), (1 / 2, 'half'), (2 / 3, 'two in three'), (3 / 4, 'three in four')]
    for f, w in words:
        if abs(v - f) < 0.02:
            return w if abs(v - f) < 0.007 else (f'nearly {w}' if v < f else f'just over {w}')
    return f'{100 * v:.0f}%'


FINDING = (
    f'From {YEARS[0]} to {YEARS[1]}, about {round(P, -2):,} people walking or biking were killed or seriously '
    f'hurt on Chicago streets; {summary["killed"]} of them died. '
    f'{share(summary["inClustersShare"]).capitalize()} of them were hurt in {n_clusters} places where it keeps '
    'happening.'
)
print(FINDING)

## 8. QA view

A plain check, not the final design: people, clusters and the HIN; clusters by size. Look for cluster
shapes that leave their streets, HIN pieces that leave their street, or a cluster name that does
not match its place.

In [ ]:
fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={'width_ratios': [1.1, 1]})
gpd.GeoSeries([city], crs=CRS).plot(ax=ax_map, color='0.93')
gpd.GeoSeries(list(shapes.values()), crs=CRS).plot(ax=ax_map, color='0.25', linewidth=2.2)
hin.plot(ax=ax_map, color='tab:orange', linewidth=1.2)
k = ppl['killed'].to_numpy()
ax_map.scatter(XY[~k, 0], XY[~k, 1], s=2, facecolor='none', edgecolor='0.45', linewidth=0.4)
ax_map.scatter(XY[k, 0], XY[k, 1], s=5, color='0.05')
for c in range(1, 11):
    p = shapes[c].interpolate(0.5, normalized=True)
    ax_map.annotate(str(c), (p.x, p.y), fontsize=8, weight='bold', color='tab:red')
minx, miny, maxx, maxy = city.bounds
ax_map.set(xlim=(minx, maxx), ylim=(miny, maxy), title='KSI people, clusters (dark), HIN (orange)')
ax_map.set_axis_off()

t = clusters.head(15).iloc[::-1]
labels = t['rank'].astype(str) + ' ' + t['name'] + ' · ' + t['community']
ax_bar.barh(labels, t['walk'], color='0.2', label='walking')
ax_bar.barh(labels, t['bike'], left=t['walk'], color='0.65', label='biking')
for i, on in enumerate(t['onHin']):
    if on:
        ax_bar.annotate('HIN', (t['people'].iloc[i] + 1, i), va='center', fontsize=7)
ax_bar.set(title='Largest clusters, people killed or seriously hurt 2021-2025')
ax_bar.legend(loc='lower right', frameon=False)
for side in ('top', 'right'):
    ax_bar.spines[side].set_visible(False)
fig.tight_layout()
plt.show()

## 9. Export

`plate.json` carries the data ledger, the finding, the summary, the clusters with their street shapes, the HIN
corridors and one record per person (position, mode, killed or not, cluster). No date, age or other
detail about a person leaves the notebook. Keys are camelCase because the website reads them directly.

In [ ]:
def line_to_address(geom: shapely.LineString, tolerance: float = 15) -> list[list[int]]:
    ll = gpd.GeoSeries([geom], crs=CRS).to_crs('EPSG:4326').iloc[0]
    xy = shapely.transform(ll, lambda c: np.column_stack(to_address(c[:, 1], c[:, 0])))
    return [[round(x), round(y)] for x, y in shapely.simplify(xy, tolerance).coords]


largest = gpd.GeoSeries([shapes[1]], crs=CRS).to_crs('EPSG:4326').total_bounds  # lon0, lat0, lon1, lat1
ix, iy = to_address([largest[1], largest[3]], [largest[0], largest[2]])
inset_box = {'west': float(ix[0]), 'east': float(ix[1]), 'south': float(iy[0]), 'north': float(iy[1])}

ledger = Ledger(
    source='Traffic Crashes - People; Traffic Crashes - Crashes',
    dataset_id=f'{PEOPLE_ID}, {CRASHES_ID}',
    url=f'{PORTAL}/d/{PEOPLE_ID}',
    pulled=pulled,
    units=f'{P:,} people walking or biking, killed or seriously injured, {YEARS[0]}-{YEARS[1]}',
    method=[
        'KSI: injury classification fatal or incapacitating (SR 1050 K and A), as recorded by police',
        f'HDBSCAN on positions in meters, leaf selection, minimum cluster {MIN_CLUSTER}, min samples {MIN_SAMPLES}',
        f'Kept: clusters that hold in resampling (stability {STABLE} or more, {BOOT_RUNS} refits on 80% samples)',
        'Shapes: the streets joining each cluster\'s people (minimum spanning tree of network distances on OSM streets)',
        f'HIN: CDOT 2025 corridors drawn on OpenStreetMap streets ({osm_date}); a person is on it within 75 ft',
    ],
    caveats=[
        'Counts, not rates: no citywide counts of people walking or biking exist',
        'Police-reported crashes on CPD streets only; expressways excluded',
        'Deaths from CDOT Vision Zero list (30-day rule); serious injuries as police recorded them',
    ],
)

r1 = lambda v: round(float(v), 1)
r3 = lambda v: round(float(v), 3)
payload = {
    'window': list(YEARS),
    'sources': {
        'people': {'id': PEOPLE_ID, 'url': f'{PORTAL}/d/{PEOPLE_ID}'},
        'crashes': {'id': CRASHES_ID, 'url': f'{PORTAL}/d/{CRASHES_ID}', 'pulled': crashes_pulled.isoformat()},
        'visionZero': {'id': VZ_ID, 'url': f'{PORTAL}/d/{VZ_ID}', 'pulled': vz_pulled.isoformat()},
        'areas': {'id': AREAS_ID, 'url': f'{PORTAL}/d/{AREAS_ID}'},
        'report': CDOT_REPORT,
        'osm': {'extract': osm_date.isoformat(), 'url': 'https://download.geofabrik.de/north-america/us/illinois.html'},
        'gtfs': {'feedDate': feed_date, 'url': 'https://www.transitchicago.com/developers/gtfs/'},
        'boundaryPulled': boundary_pulled,
    },
    'params': {'minCluster': MIN_CLUSTER, 'minSamples': MIN_SAMPLES, 'stable': STABLE, 'onHin': ON_HIN,
               'stopM': STOP_M, 'hinBufferFt': 75},
    'dropped': dropped,
    'summary': {k: r3(v) if isinstance(v, float) else v for k, v in summary.items()},
    'sensitivity': sens,
    'crossCheck': {'deaths': deaths, 'byYear': cross.reset_index(names='year').to_dict(orient='records'),
                   'report2024': report.reset_index(names='group').to_dict(orient='records')},
    'clusters': [
        {**{k: (r3(v) if isinstance(v, float) else v) for k, v in row.items()},
         'streets': [line_to_address(part, 10) for part in getattr(shapes[row['rank']], 'geoms', [shapes[row['rank']]])]}
        for row in clusters.to_dict(orient='records')
    ],
    'hin': [{'corridor': int(r.corridor), 'label': r.label, 'miles': round(r.miles, 2), 'line': line_to_address(r.geometry)}
            for r in hin.itertuples()],
    'people': [{'x': round(r.x), 'y': round(r.y), 'm': r.mode[0], 'k': int(r.killed), 'c': int(r.cluster)}
               for r in ppl.itertuples()],
    'inset': {'box': inset_box, 'cluster': 1},
}

path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)

# Read back and confirm the contract the website depends on.
written = json.loads(path.read_text())
assert {'asOf', 'finding', 'ledger', 'summary', 'clusters', 'hin', 'people', 'inset'} <= written.keys()
assert len(written['people']) == P and len(written['clusters']) == n_clusters
log.info('Wrote %s (%.1f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)

## Notes for the plate

- Finding: printed in section 7; it is generated from the data.
- Caveats for the ledger: counts are not risk (busy streets have more people to hurt); police reports
  only, on CPD streets; HIN geometry is drawn from CDOT's corridor list, not CDOT's own file.
- Next: the website reads `output/plate.json`; rebuild the site and export the PNG.